# Naive Bayes Classifier
In this notebook we will go over the creation from scratch of a Naive Bayes (NB) classifier. NB is based on Bayes' theorem, a statistical theory that calculates the probability of an outcome $P(A|B)$ based on historical probability of that outcome $P(A|B)$, the probabilities of any number of the input features occurring $P(B)$ and the probability of the outcome occurring independent of the features $P(A)$. This is according to: $P(A|B) = \frac{P(B|A)P(A)}{P(B)}$. <br>
This will be a binary classifier, so we will have to calculate the above twice, one for each class. We will see at the end that we can combine these into one identifier. <br>
NB are good for document classification in NLP, as frequency of words used can be used as features into the model. <br>
As a final note, there are functions in place to do this in packages like sklearn. However, building from scratch gives greater customisability, alongside being a pedagogical tool. A balance has to be struck between ease of programming vs customisability, so a from scratch approach may not always be the best option. Trying to make a neural network from the underlying maths would be needlessly complicated!

In [20]:
import numpy as np
import pandas as pd
import os
import re
import string
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

## Data
The data is based on a kaggle dataset of questions asked on quora. Some of these questions are sincere, and some are not. We will be trying to identify which are which.

In [2]:
train = pd.read_csv('../data/nb_train.csv')
test = pd.read_csv('../data/nb_test.csv')

In [3]:
train

,qid,question_text,target
0,00002165364db923c7e6,How did Quebec nationalists see their province...,0
1,000032939017120e6e44,"Do you have an adopted dog, how would you enco...",0
2,0000412ca6e4628ce2cf,Why does velocity affect time? Does velocity a...,0
3,000042bf85aa498cd78e,How did Otto von Guericke used the Magdeburg h...,0
4,0000455dfa3e01eae3af,Can I convert montra helicon D to a mountain b...,0
...,...,...,...
1306117,ffffcc4e2331aaf1e41e,What other technical skills do you need as a c...,0
1306118,ffffd431801e5a2f4861,Does MS in ECE have good job prospects in USA ...,0
1306119,ffffd48fb36b63db010c,Is foam insulation toxic?,0
1306120,ffffec519fa37cf60c78,How can one start a research project based on ...,0


In [4]:
test

,qid,question_text
0,0000163e3ea7c7a74cd7,Why do so many women become so rude and arroga...
1,00002bd4fb5d505b9161,When should I apply for RV college of engineer...
2,00007756b4a147d2b0b3,What is it really like to be a nurse practitio...
3,000086e4b7e1c7146103,Who are entrepreneurs?
4,0000c4c3fbe8785a3090,Is education really making good people nowadays?
...,...,...
375801,ffff7fa746bd6d6197a9,How many countries listed in gold import in in...
375802,ffffa1be31c43046ab6b,Is there an alternative to dresses on formal p...
375803,ffffae173b6ca6bfa563,Where I can find best friendship quotes in Tel...
375804,ffffb1f7f1a008620287,What are the causes of refraction of light?


Sample one from the training set

In [5]:
samp = train.sample(1)
print(samp, '\n')
sentence = samp.iloc[0].question_text
print(sentence)

                         qid  \
216455  2a57ed741f91f90662b9   

                                            question_text  target  
216455  What is the most frustrating thing about worki...       0   

What is the most frustrating thing about working with women?


## Data Preprocessing 

Remove numbers using regex. Regex stands for regular expressions and is a very efficient, albeit non-intuitive, way of understanding text. There are too many expressions to go into at this time, but a cheat sheet can be found here: https://cheatography.com/davechild/cheat-sheets/regular-expressions/

In [6]:
sentence = re.sub(r'\d+', '', sentence) 
#replaces any number of digits in `sentence` with nothing
#r at the front means raw string ie Hello\nWorld would be read literally, 
#without the line break
print('Sentence after removing numbers:', sentence)

Sentence after removing numbers: What is the most frustrating thing about working with women?


Remove punctuation

In [7]:
sentence = sentence.translate(sentence.maketrans('', '', string.punctuation))
#a bit complicated, but basically sets the punctuation to be a None type ie
#will not get counted in the string (www.programiz.com/python-programming/methods/string/maketrans)
print('Sentence after removing punct.: ', sentence)

Sentence after removing punct.:  What is the most frustrating thing about working with women


Removing stop words can be done using nltk. Stop words are those that add no particular meaning, but are required in a sentence for it to make grammatical sense. Words like: 'and', 'the', and 'a'. Here we use already in-place corpi from the `nltk` library. 

In [8]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Joe_Davies\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [9]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Joe_Davies\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

A `set()` is used to store multiple items in a single variable

In [10]:
stop_words = set(nltk.corpus.stopwords.words('english'))
words_in_sentence = list(set(sentence.split(' ')) - stop_words)
#set() allows us to efficiently remove duplicates by defining a set (group of
#immutable, unique values)
print(words_in_sentence)

['What', 'women', 'frustrating', 'thing', 'working']


We also want to get the stems of words, rather than any modified forms. We use `PorterStemmer` from `nltk.stem`

In [11]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Joe_Davies\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [12]:
nltk.download('omw-1.4')

[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Joe_Davies\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [13]:
#PorterStemmer() uses Porter stemming
stemmer = PorterStemmer()
for i, word in enumerate(words_in_sentence):
    words_in_sentence[i] = stemmer.stem(word)
print(words_in_sentence)

['what', 'women', 'frustrat', 'thing', 'work']


Lemmatization is process by which we group together the different inflected forms of the terms

In [14]:
lemmatizer = WordNetLemmatizer()
words = []
for i, word in enumerate(words_in_sentence):
    words_in_sentence[i] = lemmatizer.lemmatize(word)
print(words_in_sentence)

['what', 'woman', 'frustrat', 'thing', 'work']


# Creating the Naive Bayes Model

In [15]:
train, test = train_test_split(train, test_size=0.2)

In [16]:
word_count = {}
word_count_sincere = {}
word_count_insincere = {}
sincere = 0
insincere = 0

In [17]:
stop_words = set(nltk.corpus.stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

We have three dictionaries: word count overall, word count of sincere, and word count of insincere after preprocessing.

In [18]:
row_count = train.shape[0]

In [21]:
for row in tqdm(range(row_count)):
    target = train.iloc[row].target
    text = train.iloc[row].question_text
    insincere += target
    sincere += (1 - target)
    sentence = text
    sentence = re.sub(r'\d+', '', sentence)
    sentence = sentence.translate(sentence.maketrans('', '', string.punctuation))
    words_in_sentence = list(set(sentence.split(' ')) - stop_words)
    
    for index, word in enumerate(words_in_sentence):
            if target == 0: #0 is sincere
                if word in word_count_sincere.keys():
                    word_count_sincere[word] += 1
                else:
                    word_count_sincere[word] = 1 #set to be 1 as first instance 
            else: #insincere = 1
                if word in word_count_insincere.keys():
                    word_count_insincere[word] += 1
                else:
                    word_count_insincere[word] = 1 #same as above
            
            if word in word_count.keys(): #used to calc probability later
                word_count[word] += 1
            else:
                word_count[word] = 1

  0%|          | 0/1044897 [00:00<?, ?it/s]

We want to find the probability for each word in the dictionary, after eliminating insignificant words. We set the insignificance threshold at 0.0001.

In [22]:
word_prob = {}
total_words = 0.
for i in word_count:
    total_words += word_count[i] #gets us the total number of words in the text

#calc. prob. for each word
for i in word_count:
    word_prob[i] = word_count[i] / total_words

In [23]:
#Finding the number of non-repeated words by looking at length of word_prob
print('Total words: %d'%(len(word_prob)))
print('Min. Probability: %d'%(min(word_prob.values())))

Total words: 246260
Min. Probability: 0


In [24]:
threshold = 0.0001
for i in tqdm(list(word_prob)):
    if word_prob[i] < threshold:
        del word_prob[i] #get rid term in dictionary
        if i in list(word_count_sincere):
            del word_count_sincere[i] #remove from sincere words
        if i in list(word_count_insincere):
            del word_count_insincere[i] #as above

print('Total words passing threshold: %d'%(len(word_prob)))

  0%|          | 0/246260 [00:00<?, ?it/s]

Total words passing threshold: 1479


To find the naive bayes we must find the conditional prob..

In [25]:
tot_sincere = sum(word_count_sincere.values())
cp_sincere = {} #conditional prob.
for i in list(word_count_sincere):
    cp_sincere[i] = word_count_sincere[i] / tot_sincere

In [26]:
tot_insincere = sum(word_count_insincere.values())
cp_insincere = {} #conditional prob.
for i in list(word_count_insincere):
    cp_insincere[i] = word_count_insincere[i] / tot_insincere

Now we can use this information to make a predicition

In [27]:
row_count = test.shape[0]

p_insin = insincere / (sincere + insincere)
p_sin = sincere / (sincere + insincere)
accuracy = 0.

for row in tqdm(range(row_count)):
    target = test.iloc[row].target
    sentence = test.iloc[row].question_text
    sentence = re.sub(r'\d+', '', sentence)
    sentence = sentence.translate(sentence.maketrans('', '', string.punctuation))
    words_in_sentence = list(set(sentence.split(' ')) - stop_words)
    for index, word in enumerate(words_in_sentence):
        word = stemmer.stem(word)
        words_in_sentence[index] = lemmatizer.lemmatize(word)
    
    insin_term = p_insin
    sin_term = p_sin
    
    sin_M = len(cp_sincere.keys()) #num insin in total
    insin_M = len(cp_insincere.keys()) #as above sincere
    for word in words_in_sentence:
        if word not in cp_insincere.keys():
            insin_M += 1
        if word not in cp_sincere.keys():
            sin_M += 1
    
    for word in words_in_sentence:
        if word in cp_insincere.keys():
            insin_term *= (cp_insincere[word] + (1/insin_M)) #P(insin)*P(insin|all_words)/P(insin_num_words) 
            #insin prob for whole thing * conditional prob of word 
            #+ prob a word is insin in current text
        else:
            insin_term *= (1/insin_M)
        if word in cp_sincere.keys():
            sin_term *= (cp_sincere[word] + (1/sin_M))
        else:
            sin_term *= (1/sin_M)
        
    if insin_term/(insin_term + sin_term) > 0.5: #classification (softmax kinda)
        response = 1
    else:
        response = 0
    if target == response:
        accuracy += 1
    
print ('Accuracy is ',accuracy/row_count*100)

  0%|          | 0/261225 [00:00<?, ?it/s]

Accuracy is  93.99444922959135


In [28]:
insin_term

1.9943488912622207e-23

In [29]:
sin_term

5.997924289889303e-22